# BrailleLens — lighting-robust cell detection & classification (no new photos)

Fine-tunes the two models the app uses for the page scan so they cope with **dim light and side light**, using only the
**existing 12 Gold pages** (each has a normal and a low-light photo, with LabelMe cell boxes). No new photo shoot needed.

| Model in the app | Starting checkpoint | What this notebook adds |
|---|---|---|
| Cell detector `braille_cell_yolo26n_degraded_mobile.onnx` | `braille_cell_gold_degraded.pt` | Lighting augmentation, mirror flips, **no-Braille negative images** |
| Cell classifier `braille_cnn.onnx` | `braille_cnn_gold_finetuned.pt` | Lighting augmentation, **flip + dot-relabel**, consistency loss, a **"not a cell" class** |

**How lighting is simulated (no new images)**
1. **Relief relighting:** an embossed dot's shading is roughly proportional to `slope · light direction`. Negating the
   image's high-pass detail ≈ light from the *opposite* side; scaling it ≈ low raking light (long shadows) vs flat light.
2. **Mirror flips:** a mirrored photo of cell *c* lit from the left is exactly a photo of the mirrored cell lit from the right.
   For the classifier the label is remapped (horizontal: dots 1↔4, 2↔5, 3↔6; vertical: 1↔3, 4↔6).
3. **Illumination:** directional brightness ramps, vignetting, soft cast shadows, colour temperature.
4. **Sensor:** low exposure + shot/read noise + auto-gain (what a phone does in a dark room), blur, JPEG,
   and the calibrated glasses-camera softening (0.39× down/up, from `cell_detect/CAMERA_DEGRADATION.md`).

**How false cells on Braille-free images are prevented**
- **Detector:** background images with empty labels (Gold pages with every cell *inpainted out*, COCO scenes,
  anything in `negatives_extra/`), and a confidence threshold chosen on validation data that includes negatives.
- **Classifier:** class 64 = "not a cell" (paper, gaps, half-shifted boxes, clutter). The app already drops any class
  without a Sinhala label (`'#64'` → dropped), so **no Dart change is needed** for this class.
- **Page-level gate:** a real page has many same-sized, taller-than-wide boxes in rows. The notebook measures a simple
  gate (min count + size consistency + row structure) you can later add in the app.

**Honest evaluation:** held-out Gold pages 10 & 11 (normal + real low-light photos), a synthetic corruption benchmark
at 3 severities (with one corruption family, *glare*, never used in training), Braille-free test images, and raw
glasses-camera frames for a visual check. Old vs new models are compared on all of them. **Nothing is switched in the app.**

### Before running
1. On the PC: `finger_cell_track\.venv\Scripts\python.exe lighting_robust/pack_for_colab.py`
2. Upload `colab_upload/lighting_robust_bundle.zip` to **`MyDrive/BrailleLens_LightingRobust/`**
3. Runtime → **T4 GPU** → Run all (~1–1.5 h).

If Drive mounting fails with *"credential propagation was unsuccessful"*: rerun the setup cell and tick **all** permission
boxes in the Google popup (same account as Colab; allow popups / third-party cookies for colab.research.google.com).
Or skip Drive: upload the zip to `/content/` with the Files panel; outputs then go to `/content/outputs` (download them
before the session ends).

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim onnxscript opencv-python-headless

## 1) Setup and configuration

In [ ]:
import json, math, random, shutil, sys, time, zipfile
from pathlib import Path

import cv2
import numpy as np
import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
from google.colab import drive

# Drive is preferred (outputs survive a disconnect). If mounting fails ("credential propagation was unsuccessful"),
# upload lighting_robust_bundle.zip with the Files panel (left sidebar) into /content/ and rerun this cell.
LOCAL_BUNDLE = Path("/content/lighting_robust_bundle.zip")
try:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/BrailleLens_LightingRobust")
except Exception as e:
    print(f"Drive mount failed ({e}); falling back to /content (download outputs/ before the session ends)")
    DRIVE = Path("/content")
BUNDLE = DRIVE / "lighting_robust_bundle.zip"
if not BUNDLE.exists() and LOCAL_BUNDLE.exists():
    BUNDLE = LOCAL_BUNDLE
OUT = DRIVE / "outputs"
WORK = Path("/content/lr")
assert BUNDLE.exists(), (f"Bundle not found. Put it at {DRIVE / 'lighting_robust_bundle.zip'} "
                         f"or upload it to {LOCAL_BUNDLE} via the Files panel")
print("bundle:", BUNDLE, "| outputs:", OUT)
OUT.mkdir(parents=True, exist_ok=True)
if not (WORK / "gold").exists():
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(WORK)
sys.path.insert(0, str(WORK / "code"))

# Same split as the current models (cell_detect/CAMERA_DEGRADATION.md): train 1-8, val 9 & 12, test 10 & 11.
TRAIN_PAGES, VAL_PAGES, TEST_PAGES = [1, 2, 3, 4, 5, 6, 7, 8], [9, 12], [10, 11]
# Low-light photos of train pages used in training (1-4 is the proven set; 5-8 regressed earlier runs).
TRAIN_LOW_PAGES = [1, 2, 3, 4]

DET_AUG_VARIANTS = 6        # augmented copies per training page photo
DET_NEG_FRACTION = 0.20     # negatives as a fraction of positive training images
DET_EPOCHS, DET_IMGSZ, DET_BATCH = 60, 1280, 4
CNN_EPOCHS, CNN_BATCH = 30, 256
CNN_REJECT_RATIO = 0.6      # "not a cell" crops per real cell
HELDOUT_FAMILIES = ["glare"]  # never used in training -> unseen-corruption score
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DET_BASE = WORK / "weights" / "braille_cell_gold_degraded.pt"
CNN_BASE = WORK / "weights" / "braille_cnn_gold_finetuned.pt"
for p in (DET_BASE, CNN_BASE):
    assert p.exists(), p
print("bundle ok:", sorted(x.name for x in WORK.iterdir()))

## 2) Load the Gold pages

In [ ]:
def dots_to_code(label: str) -> int:
    label = str(label).strip()
    return 0 if label == "0" else sum(1 << (int(d) - 1) for d in set(label))

def load_page(variant: str, n: int):
    js = WORK / "gold" / variant / f"pg-{n}.json"
    if not js.exists():
        return None
    img_path = next((js.with_suffix(e) for e in (".jpeg", ".jpg", ".png") if js.with_suffix(e).exists()), None)
    doc = json.loads(js.read_text(encoding="utf-8"))
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img.shape[1] != doc["imageWidth"] or img.shape[0] != doc["imageHeight"]:
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    assert img.shape[1] == doc["imageWidth"] and img.shape[0] == doc["imageHeight"], js
    boxes, codes = [], []
    for s in doc["shapes"]:
        if s.get("shape_type") != "rectangle":
            continue
        (a, b), (c, d) = s["points"][:2]
        boxes.append([min(a, c), min(b, d), max(a, c), max(b, d)])
        codes.append(dots_to_code(s["label"]))
    return dict(variant=variant, page=n, img=img,
                boxes=np.array(boxes, np.float32), codes=np.array(codes, np.int64))

PAGES = {}
for v in ("high", "low"):
    for n in range(1, 13):
        rec = load_page(v, n)
        if rec is not None:
            PAGES[(v, n)] = rec

def split_pages(split):
    pages = {"train": TRAIN_PAGES, "val": VAL_PAGES, "test": TEST_PAGES}[split]
    out = [PAGES[("high", n)] for n in pages if ("high", n) in PAGES]
    lows = TRAIN_LOW_PAGES if split == "train" else pages
    out += [PAGES[("low", n)] for n in lows if ("low", n) in PAGES]
    return out

for s in ("train", "val", "test"):
    ps = split_pages(s)
    print(f"{s:5s}: {len(ps)} photos, {sum(len(p['boxes']) for p in ps)} cells  "
          f"{[(p['variant'][0] + str(p['page'])) for p in ps]}")

## 3) Lighting augmentation library
Each function takes a uint8 image (page BGR or crop gray) and returns uint8. `sigma` is the relief scale
(≈ dot size in pixels). The same functions drive training augmentation and the evaluation benchmark.

In [ ]:
def _f(x): return x.astype(np.float32)
def _u8(x): return np.clip(x, 0, 255).astype(np.uint8)
def _gain(img, g): return _u8(_f(img) * (g[..., None] if img.ndim == 3 else g))

def relief(img, rng, sigma, strength=None, flip=None):
    """Re-light embossed dots: scale (raking vs flat light) and optionally negate (opposite light side)."""
    base = cv2.GaussianBlur(_f(img), (0, 0), sigma)
    high = _f(img) - base
    s = rng.uniform(0.4, 1.9) if strength is None else strength
    if (rng.random() < 0.5) if flip is None else flip:
        s = -s
    return _u8(base + high * s)

def directional(img, rng, strength=None):
    h, w = img.shape[:2]
    th = rng.uniform(0, 2 * np.pi)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    proj = ((xx - w / 2) * np.cos(th) + (yy - h / 2) * np.sin(th)) / (0.5 * np.hypot(w, h))
    lo = 1 - (rng.uniform(0.3, 0.85) if strength is None else strength)
    g = (lo + (1 - lo) * np.clip((proj + 1) / 2, 0, 1)) ** rng.uniform(0.8, 1.6)
    return _gain(img, g)

def vignette(img, rng, strength=None):
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    r = np.hypot((xx - w / 2) / (w / 2), (yy - h / 2) / (h / 2)) / np.sqrt(2)
    k = rng.uniform(0.2, 0.6) if strength is None else strength
    return _gain(img, 1 - k * r ** 2)

def cast_shadow(img, rng, darkness=None):
    h, w = img.shape[:2]
    mask = np.zeros((h, w), np.float32)
    for _ in range(rng.integers(1, 3)):
        cx, cy = rng.uniform(0, w), rng.uniform(0, h)
        rad = rng.uniform(0.15, 0.5) * max(h, w)
        n = rng.integers(5, 9)
        ang = np.sort(rng.uniform(0, 2 * np.pi, n))
        rr = rad * rng.uniform(0.5, 1.2, n)
        pts = np.stack([cx + rr * np.cos(ang), cy + rr * np.sin(ang)], 1).astype(np.int32)
        cv2.fillPoly(mask, [pts], 1.0)
    mask = cv2.GaussianBlur(mask, (0, 0), max(h, w) * rng.uniform(0.02, 0.08))
    d = rng.uniform(0.3, 0.65) if darkness is None else darkness
    return _gain(img, 1 - d * np.clip(mask, 0, 1))

def color_temp(img, rng):
    if img.ndim != 3:
        return img
    warm = rng.uniform(-1, 1)
    g = np.array([1 - 0.25 * warm, 1.0, 1 + 0.15 * warm], np.float32)  # BGR
    return _u8(_f(img) * g)

def low_light(img, rng, exposure=None):
    """Dark room: few photons -> shot + read noise -> camera auto-gain amplifies it -> optional denoise smear."""
    k = rng.uniform(0.1, 0.5) if exposure is None else exposure
    x = _f(img) / 255.0 * k
    a, b = rng.uniform(0.004, 0.014), rng.uniform(1e-5, 3e-4)
    x = x + rng.standard_normal(x.shape).astype(np.float32) * np.sqrt(np.maximum(x * a + b, 1e-8))
    x = np.clip(x * (1 / k) * rng.uniform(0.5, 0.95), 0, 1) ** rng.uniform(0.85, 1.3)
    out = _u8(x * 255)
    if rng.random() < 0.5:
        out = cv2.GaussianBlur(out, (0, 0), rng.uniform(0.5, 1.2))
    return out

def blur(img, rng, scale=1.0, amount=None):
    if rng.random() < 0.5:
        k = int(round((rng.uniform(3, 9) if amount is None else amount * 3) * scale)) | 1
        kern = np.zeros((k, k), np.float32)
        kern[k // 2, :] = 1.0 / k
        rot = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), rng.uniform(0, 180), 1.0)
        kern = cv2.warpAffine(kern, rot, (k, k))
        kern /= max(kern.sum(), 1e-6)
        return cv2.filter2D(img, -1, kern)
    s = (rng.uniform(0.8, 2.2) if amount is None else amount) * scale
    return cv2.GaussianBlur(img, (0, 0), s)

def glasses_soft(img, rng, factor=0.39):
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(2, int(w * factor)), max(2, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)

def jpeg(img, rng, q=None):
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, int(rng.uniform(25, 75) if q is None else q)])
    return cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)

def glare(img, rng, strength=None):
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    cx, cy = rng.uniform(0.2, 0.8) * w, rng.uniform(0.2, 0.8) * h
    rad = rng.uniform(0.1, 0.3) * max(h, w)
    blob = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * rad ** 2))
    s = rng.uniform(60, 140) if strength is None else strength
    add = blob * s
    return _u8(_f(img) + (add[..., None] if img.ndim == 3 else add))

def random_lighting(img, rng, sigma, scale=1.0, exclude=()):
    """Training-time composition: scene relighting -> illumination -> colour -> sensor -> optics -> codec."""
    ops = [
        ("relief", 0.8, lambda x: relief(x, rng, sigma)),
        ("directional", 0.7, lambda x: directional(x, rng)),
        ("vignette", 0.3, lambda x: vignette(x, rng)),
        ("shadow", 0.35, lambda x: cast_shadow(x, rng)),
        ("color_temp", 0.4, lambda x: color_temp(x, rng)),
        ("low_light", 0.45, lambda x: low_light(x, rng)),
        ("blur", 0.35, lambda x: blur(x, rng, scale)),
        ("glasses", 0.15, lambda x: glasses_soft(x, rng)),
        ("jpeg", 0.4, lambda x: jpeg(x, rng)),
        ("glare", 0.2, lambda x: glare(x, rng)),
    ]
    for name, p, fn in ops:
        if name not in exclude and rng.random() < p:
            img = fn(img)
    return img

# Fixed-severity corruptions for the benchmark (seeded, identical for every model).
def corrupt(img, family, sev, seed, sigma):
    rng = np.random.default_rng(seed)
    s = sev - 1
    if family == "dark":
        return low_light(img, rng, exposure=[0.4, 0.22, 0.1][s])
    if family == "side_light":
        x = relief(img, rng, sigma, strength=[1.4, 1.7, 2.0][s], flip=bool(seed % 2))
        return directional(x, rng, strength=[0.4, 0.6, 0.8][s])
    if family == "shadow":
        return cast_shadow(img, rng, darkness=[0.35, 0.5, 0.65][s])
    if family == "blur":
        return cv2.GaussianBlur(img, (0, 0), [0.8, 1.4, 2.0][s] * img.shape[1] / 960)
    if family == "glasses":
        return glasses_soft(img, rng, factor=[0.5, 0.39, 0.3][s])
    if family == "color_temp":
        return color_temp(img, rng)
    if family == "glare":
        return glare(img, rng, strength=[70, 110, 150][s])
    raise ValueError(family)

FAMILIES = ["dark", "side_light", "shadow", "blur", "glasses", "color_temp", "glare"]
PAGE_SIGMA = 3.0  # relief scale for a 960-px-wide page (dots are ~4-6 px)

# Visual sanity check
demo = PAGES[("high", 1)]["img"]
rng = np.random.default_rng(0)
tiles = [demo] + [random_lighting(demo.copy(), rng, PAGE_SIGMA) for _ in range(5)]
tiles += [corrupt(demo, f, 3, 1, PAGE_SIGMA) for f in FAMILIES[:6]]
y0, x0 = 300, 200
grid = [cv2.resize(t[y0:y0 + 400, x0:x0 + 500], (500, 400)) for t in tiles]
sheet = np.vstack([np.hstack(grid[i:i + 6]) for i in (0, 6)])
cv2.imwrite(str(OUT / "augmentation_preview.jpg"), sheet)
from IPython.display import Image as _Img, display
display(_Img(str(OUT / "augmentation_preview.jpg"), width=1400))
print("top: original + 5 random training augmentations | bottom: benchmark at severity 3:", FAMILIES[:6])

## 4) Negative images (no Braille)
- **Inpainted Gold pages:** every labelled cell removed → same paper, lighting and camera, but zero cells. These are the
  hardest negatives (paper texture, shadows, page edges, spine).
- **COCO-128 scenes:** desks, hands, objects, printed text.
- **`negatives_extra/`:** any Braille-free photos you added before packing (optional).

Train pages' inpainted copies go to training; val/test pages' copies stay in val/test negatives.

In [ ]:
def inpaint_cells(rec, grow=0.3):
    img = rec["img"]
    mask = np.zeros(img.shape[:2], np.uint8)
    for x0, y0, x1, y1 in rec["boxes"]:
        gw, gh = (x1 - x0) * grow, (y1 - y0) * grow
        cv2.rectangle(mask, (int(x0 - gw), int(y0 - gh)), (int(x1 + gw), int(y1 + gh)), 255, -1)
    return cv2.inpaint(img, mask, 7, cv2.INPAINT_TELEA)

coco_dir = WORK / "coco128"
if not coco_dir.exists():
    import urllib.request
    urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip", "/content/coco128.zip")
    with zipfile.ZipFile("/content/coco128.zip") as z:
        z.extractall(WORK)
coco = sorted((coco_dir / "images" / "train2017").glob("*.jpg"))
random.Random(SEED).shuffle(coco)
COCO_TRAIN, COCO_VAL, COCO_TEST = coco[:88], coco[88:104], coco[104:]
extra = sorted(p for p in (WORK / "negatives_extra").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})
ext_tr, ext_te = extra[: len(extra) // 2], extra[len(extra) // 2:]

NEG = {s: [] for s in ("train", "val", "test")}
for s in ("train", "val", "test"):
    for rec in split_pages(s):
        NEG[s].append(("inpaint", f"{rec['variant']}{rec['page']}", inpaint_cells(rec)))
for s, files in (("train", COCO_TRAIN + ext_tr), ("val", COCO_VAL), ("test", COCO_TEST + ext_te)):
    for f in files:
        im = cv2.imread(str(f))
        if im is not None:
            NEG[s].append(("coco" if "coco" in str(f) else "extra", f.stem, im))
print({s: len(v) for s, v in NEG.items()})

## 5) Build the detector dataset

In [ ]:
DET = WORK / "det"
if DET.exists():
    shutil.rmtree(DET)

def write_yolo(img, boxes, split_dir, stem):
    (split_dir / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "labels").mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(split_dir / "images" / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    h, w = img.shape[:2]
    lines = []
    for x0, y0, x1, y1 in boxes:
        x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, w), min(y1, h)
        if x1 - x0 >= 1 and y1 - y0 >= 1:
            lines.append(f"0 {(x0 + x1) / 2 / w:.6f} {(y0 + y1) / 2 / h:.6f} {(x1 - x0) / w:.6f} {(y1 - y0) / h:.6f}")
    (split_dir / "labels" / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))

def flip_boxes(boxes, w, h, hflip, vflip):
    b = boxes.copy()
    if hflip:
        b[:, [0, 2]] = w - boxes[:, [2, 0]]
    if vflip:
        b[:, [1, 3]] = h - boxes[:, [3, 1]]
    return b

rng = np.random.default_rng(SEED)
n_pos = 0
for rec in split_pages("train"):
    stem = f"{rec['variant']}{rec['page']}"
    write_yolo(rec["img"], rec["boxes"], DET / "train", stem); n_pos += 1
    h, w = rec["img"].shape[:2]
    for k in range(DET_AUG_VARIANTS):
        hf, vf = rng.random() < 0.5, rng.random() < 0.3
        img = rec["img"][:, ::-1] if hf else rec["img"]
        img = img[::-1] if vf else img
        img = random_lighting(np.ascontiguousarray(img), rng, PAGE_SIGMA, exclude=HELDOUT_FAMILIES)
        write_yolo(img, flip_boxes(rec["boxes"], w, h, hf, vf), DET / "train", f"{stem}_aug{k}"); n_pos += 1

neg_budget = int(round(n_pos * DET_NEG_FRACTION))
neg_pool = list(NEG["train"])
rng.shuffle(neg_pool)
for i in range(neg_budget):
    kind, name, im = neg_pool[i % len(neg_pool)]
    if i >= len(neg_pool) or rng.random() < 0.6:  # repeats are always re-augmented
        im = random_lighting(im.copy(), rng, PAGE_SIGMA * im.shape[1] / 960, exclude=HELDOUT_FAMILIES)
    write_yolo(im, np.zeros((0, 4)), DET / "train", f"neg_{kind}_{name}_{i}")

for rec in split_pages("val"):
    write_yolo(rec["img"], rec["boxes"], DET / "val", f"{rec['variant']}{rec['page']}")
for i, (kind, name, im) in enumerate(NEG["val"]):
    write_yolo(im, np.zeros((0, 4)), DET / "val", f"neg_{kind}_{name}_{i}")

# Evaluation sets (kept in memory as (name, image, gt_boxes) for the custom evaluator)
EVAL = {"clean": [], "neg": []}
for rec in split_pages("test"):
    EVAL["clean"].append((f"{rec['variant']}{rec['page']}", rec["img"], rec["boxes"]))
for fam in FAMILIES:
    for sev in (1, 2, 3):
        EVAL[f"{fam}_s{sev}"] = [
            (f"{rec['variant']}{rec['page']}", corrupt(rec["img"], fam, sev, 1000 * sev + rec["page"], PAGE_SIGMA), rec["boxes"])
            for rec in split_pages("test")]
for i, (kind, name, im) in enumerate(NEG["test"]):
    EVAL["neg"].append((f"{kind}_{name}", im, np.zeros((0, 4), np.float32)))

(DET / "data.yaml").write_text(f"path: {DET}\ntrain: train/images\nval: val/images\nnc: 1\nnames:\n  0: braille_cell\n")
print(f"train: {n_pos} positive + {neg_budget} negative images; val: {len(split_pages('val'))} pages + {len(NEG['val'])} negatives")
print("eval sets:", {k: len(v) for k, v in EVAL.items()})

## 6) Fine-tune the detector
Starts from the detector the app uses now. Low learning rate (optimizer set explicitly — with `optimizer="auto"`
Ultralytics ignores `lr0`), no mosaic (too few pages), Ultralytics' own
flips off (flips are done offline above, with correct boxes), its colour jitter kept as an extra layer.

In [ ]:
from ultralytics import YOLO

det_runs = OUT / "runs_detector"
det_model = YOLO(str(DET_BASE))
det_model.train(
    data=str(DET / "data.yaml"), epochs=DET_EPOCHS, imgsz=DET_IMGSZ, batch=DET_BATCH, device=0,
    workers=2, patience=20, seed=SEED, optimizer="AdamW", lr0=0.001, cos_lr=True, max_det=800,
    mosaic=0.0, mixup=0.0, fliplr=0.0, flipud=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, degrees=3.0, translate=0.1, scale=0.2, shear=1.0, perspective=0.0005,
    project=str(det_runs), name="lighting", exist_ok=True,
)
DET_NEW = det_runs / "lighting" / "weights" / "best.pt"
assert DET_NEW.exists(), DET_NEW
shutil.copy2(DET_NEW, OUT / "braille_cell_lighting.pt")
print("saved", OUT / "braille_cell_lighting.pt")

## 7) Evaluate detectors: old vs new
Mirrors the app: predict, NMS at IoU 0.40, then keep boxes above a confidence threshold. A prediction is correct when it
overlaps an unmatched labelled cell with IoU ≥ 0.5. On Braille-free images every box is a false positive.

In [ ]:
import torchvision

def predict(model, images, conf=0.05):
    out = []
    for i in range(0, len(images), 4):
        res = model.predict([im for im in images[i:i + 4]], imgsz=DET_IMGSZ, conf=conf, max_det=800, verbose=False, device=0)
        for r in res:
            b = r.boxes.xyxy.cpu(); s = r.boxes.conf.cpu()
            keep = torchvision.ops.nms(b, s, 0.40) if len(b) else torch.zeros(0, dtype=torch.long)
            out.append((b[keep].numpy(), s[keep].numpy()))
    return out

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

def score_at(preds, gts, thr):
    tp = fp = fn = 0
    for (b, s), g in zip(preds, gts):
        b = b[s >= thr]; s = s[s >= thr]
        order = np.argsort(-s); b = b[order]
        m = iou_matrix(b, g); used = np.zeros(len(g), bool)
        for i in range(len(b)):
            j = int(np.argmax(m[i])) if len(g) else -1
            if j >= 0 and m[i, j] >= 0.5 and not used[j]:
                used[j] = True; tp += 1
            else:
                fp += 1
        fn += int((~used).sum())
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return dict(P=p, R=r, F1=2 * p * r / max(p + r, 1e-9), fp_per_img=fp / max(len(gts), 1))

DET_MODELS = {"old": YOLO(str(DET_BASE)), "new": YOLO(str(DET_NEW))}
THRS = np.round(np.arange(0.10, 0.71, 0.05), 2)

# Pick each model's threshold on VALIDATION pages + negatives (max F1), never on test.
val_items = [(r["img"], r["boxes"]) for r in split_pages("val")] + [(im, np.zeros((0, 4))) for _, _, im in NEG["val"]]
DET_THR = {}
for name, m in DET_MODELS.items():
    pv = predict(m, [x for x, _ in val_items])
    best = max(THRS, key=lambda t: score_at(pv, [g for _, g in val_items], t)["F1"])
    DET_THR[name] = float(best)
print("chosen thresholds (val F1, negatives included):", DET_THR, "| app currently uses 0.25")

PRED = {name: {k: predict(m, [im for _, im, _ in v]) for k, v in EVAL.items()} for name, m in DET_MODELS.items()}

rows = []
for k, v in EVAL.items():
    gts = [g for _, _, g in v]
    row = {"set": k}
    for name in DET_MODELS:
        for tag, thr in (("@0.25", 0.25), ("@best", DET_THR[name])):
            sc = score_at(PRED[name][k], gts, thr)
            if k == "neg":
                row[f"{name}{tag} FP/img"] = round(sc["fp_per_img"], 2)
                row[f"{name}{tag} imgs with any box"] = sum(int((s >= thr).sum() > 0) for _, s in PRED[name][k])
            else:
                row[f"{name}{tag} F1"] = round(sc["F1"], 3); row[f"{name}{tag} R"] = round(sc["R"], 3)
    rows.append(row)

import pandas as pd
det_table = pd.DataFrame(rows).set_index("set")
pd.set_option("display.width", 250)
display(det_table)
det_table.to_csv(OUT / "detector_comparison.csv")

fam_avg = {name: {fam: np.mean([score_at(PRED[name][f"{fam}_s{s}"], [g for _, _, g in EVAL[f"{fam}_s{s}"]], DET_THR[name])["F1"]
                                for s in (1, 2, 3)]) for fam in FAMILIES} for name in DET_MODELS}
display(pd.DataFrame(fam_avg).round(3).rename_axis("corruption (mean F1 over 3 severities, own best thr)"))
print("glare is held out of training -> its row shows robustness to an unseen lighting effect")

### Page-level gate
A real Braille page gives many boxes of similar size, taller than wide, lined up in rows. Thresholds come from the
**training** labels; the gate is then checked on test pages (clean + corrupted) and on Braille-free images.

In [ ]:
tr_boxes = np.concatenate([r["boxes"] for r in split_pages("train")])
tr_h = tr_boxes[:, 3] - tr_boxes[:, 1]; tr_w = tr_boxes[:, 2] - tr_boxes[:, 0]
ASPECT_LO, ASPECT_HI = np.percentile(tr_h / tr_w, [2, 98])
print(f"cell aspect h/w in training labels: 2-98th pct = {ASPECT_LO:.2f}-{ASPECT_HI:.2f}")

def page_gate(boxes, min_cells=8):
    if len(boxes) < min_cells:
        return False
    w = boxes[:, 2] - boxes[:, 0]; h = boxes[:, 3] - boxes[:, 1]
    med_h = np.median(h)
    size_ok = np.mean(np.abs(h - med_h) < 0.35 * med_h) >= 0.7
    aspect_ok = np.mean((h / np.maximum(w, 1e-6) >= ASPECT_LO) & (h / np.maximum(w, 1e-6) <= ASPECT_HI)) >= 0.7
    yc = np.sort((boxes[:, 1] + boxes[:, 3]) / 2)
    rows_, cur = [], [yc[0]]
    for y in yc[1:]:
        if y - cur[-1] < 0.5 * med_h:
            cur.append(y)
        else:
            rows_.append(cur); cur = [y]
    rows_.append(cur)
    in_rows = sum(len(r) for r in rows_ if len(r) >= 3) / len(yc)
    return bool(size_ok and aspect_ok and in_rows >= 0.6)

gate_rows = []
for name in DET_MODELS:
    thr = DET_THR[name]
    kept = lambda k: [b[s >= thr] for b, s in PRED[name][k]]
    pages_pass = np.mean([page_gate(b) for k in EVAL if k != "neg" for b in kept(k)])
    neg_pass = np.mean([page_gate(b) for b in kept("neg")])
    gate_rows.append({"model": name, "Braille pages accepted": round(pages_pass, 3), "Braille-free images accepted": round(neg_pass, 3)})
display(pd.DataFrame(gate_rows).set_index("model"))

## 8) Build the classifier dataset
Each cell is stored with surrounding context so lighting augmentation acts on real paper around the dots, then the box is
cut out (with jitter, as a detector box would be) and resized to 64×64 exactly like the app. Class **64 = not a cell**:
half-shifted boxes straddling two cells, empty paper and line gaps, inpainted-page patches, and COCO clutter.

In [ ]:
from labels import code_to_label  # braille_cnn/labels.py, shipped in the bundle's code/ folder

NC, REJECT = 65, 64
MAPPED = {c for c in range(64) if c != 0 and code_to_label(c, lang="si").strip() and not code_to_label(c, lang="si").startswith("#")}
print(f"{len(MAPPED)} codes have a Sinhala label (the rest are dropped by the app as background)")

def _perm(pairs):
    p = list(range(64))
    for c in range(64):
        out = c
        for a, b in pairs:
            ba, bb = (c >> (a - 1)) & 1, (c >> (b - 1)) & 1
            out &= ~((1 << (a - 1)) | (1 << (b - 1)))
            out |= (bb << (a - 1)) | (ba << (b - 1))
        p[c] = out
    return p + [REJECT]

HMAP = _perm([(1, 4), (2, 5), (3, 6)])   # mirror left-right
VMAP = _perm([(1, 3), (4, 6)])           # mirror top-bottom
assert HMAP[dots_to_code("1")] == dots_to_code("4") and VMAP[dots_to_code("12")] == dots_to_code("23")

def context_item(gray, box, code, ctx=1.0):
    x0, y0, x1, y1 = box
    bw, bh = x1 - x0, y1 - y0
    H, W = gray.shape
    cx0, cy0 = int(max(0, x0 - ctx * bw)), int(max(0, y0 - ctx * bh))
    cx1, cy1 = int(min(W, x1 + ctx * bw)), int(min(H, y1 + ctx * bh))
    if cx1 - cx0 < 4 or cy1 - cy0 < 4:
        return None
    return dict(patch=gray[cy0:cy1, cx0:cx1].copy(), box=(x0 - cx0, y0 - cy0, x1 - cx0, y1 - cy0), code=int(code))

def reject_boxes(rec, rng, n):
    """Cell-sized boxes that are NOT a cell: half-shifted (straddling) or random paper/background."""
    boxes = rec["boxes"]; H, W = rec["img"].shape[:2]
    ws, hs = boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]
    out, tries = [], 0
    while len(out) < n and tries < n * 30:
        tries += 1
        if rng.random() < 0.5:
            i = rng.integers(len(boxes)); b = boxes[i].copy()
            if rng.random() < 0.6:
                d = rng.uniform(0.4, 0.6) * ws[i] * rng.choice([-1, 1]); b[[0, 2]] += d
            else:
                d = rng.uniform(0.35, 0.55) * hs[i] * rng.choice([-1, 1]); b[[1, 3]] += d
            max_iou = 0.3
        else:
            i = rng.integers(len(boxes)); bw, bh = ws[i], hs[i]
            x0, y0 = rng.uniform(0, W - bw), rng.uniform(0, H - bh)
            b = np.array([x0, y0, x0 + bw, y0 + bh], np.float32)
            max_iou = 0.1
        if b[0] < 0 or b[1] < 0 or b[2] > W or b[3] > H:
            continue
        if iou_matrix(b[None], boxes).max() < max_iou:
            out.append(b)
    return out

def build_items(split, rng):
    items = []
    for rec in split_pages(split):
        gray = cv2.cvtColor(rec["img"], cv2.COLOR_BGR2GRAY)
        for b, c in zip(rec["boxes"], rec["codes"]):
            it = context_item(gray, b, c)
            if it: items.append(it)
        n_rej = int(len(rec["boxes"]) * CNN_REJECT_RATIO * 0.7)
        for b in reject_boxes(rec, rng, n_rej):
            it = context_item(gray, b, REJECT)
            if it: items.append(it)
    n_cells = sum(i["code"] != REJECT for i in items)
    neg_imgs = [im for kind, _, im in NEG[split]]
    ref_sizes = np.concatenate([r["boxes"][:, 2:] - r["boxes"][:, :2] for r in split_pages(split)])
    for _ in range(int(n_cells * CNN_REJECT_RATIO * 0.3)):
        im = cv2.cvtColor(neg_imgs[rng.integers(len(neg_imgs))], cv2.COLOR_BGR2GRAY)
        bw, bh = ref_sizes[rng.integers(len(ref_sizes))] * rng.uniform(0.7, 1.5)
        H, W = im.shape
        if W <= bw or H <= bh:
            continue
        x0, y0 = rng.uniform(0, W - bw), rng.uniform(0, H - bh)
        it = context_item(im, (x0, y0, x0 + bw, y0 + bh), REJECT)
        if it: items.append(it)
    return items

rng = np.random.default_rng(SEED)
ITEMS = {s: build_items(s, rng) for s in ("train", "val", "test")}
for s, v in ITEMS.items():
    cells = sum(i["code"] != REJECT for i in v)
    print(f"{s:5s}: {cells} cells ({len({i['code'] for i in v if i['code'] != REJECT})} codes) + {len(v) - cells} not-a-cell crops")

## 9) Classifier dataset, model and training
- Normalisation is **identical to the app** (`(px − mean) / (max(std, 10) · 4) + 0.5`), so the exported model drops in.
- Two augmented views of every crop + **consistency loss** (Jensen–Shannon, AugMix-style): the prediction must not change
  when only the lighting changes. Label smoothing 0.05 keeps confidences calibrated for the "uncertain" threshold.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

def normalize_crop(arr, std_floor=10.0, span_std=4.0):
    arr = arr.astype(np.float32); m, s = arr.mean(), arr.std()
    return np.clip((arr - m) / (max(s, std_floor) * span_std) + 0.5, 0, 1).astype(np.float32)

def cut(patch, box, margin=0.0, sx=0.0, sy=0.0):
    x0, y0, x1, y1 = box; bw, bh = x1 - x0, y1 - y0
    H, W = patch.shape
    a = int(round(np.clip(x0 - margin * bw + sx, 0, W - 2))); b = int(round(np.clip(y0 - margin * bh + sy, 0, H - 2)))
    c = int(round(np.clip(x1 + margin * bw + sx, a + 1, W))); d = int(round(np.clip(y1 + margin * bh + sy, b + 1, H)))
    return cv2.resize(patch[b:d, a:c], (64, 64), interpolation=cv2.INTER_CUBIC)

def crop_lighting(patch, rng, sigma):
    ops = [("relief", 0.8, lambda x: relief(x, rng, sigma)),
           ("directional", 0.5, lambda x: directional(x, rng, strength=rng.uniform(0.1, 0.5))),
           ("shadow", 0.25, lambda x: cast_shadow(x, rng, darkness=rng.uniform(0.2, 0.5))),
           ("low_light", 0.45, lambda x: low_light(x, rng)),
           ("contrast", 0.4, lambda x: _u8((_f(x) - x.mean()) * rng.uniform(0.3, 0.9) + x.mean())),
           ("blur", 0.3, lambda x: blur(x, rng, scale=sigma / PAGE_SIGMA * 0.6)),
           ("glasses", 0.15, lambda x: glasses_soft(x, rng)),
           ("jpeg", 0.35, lambda x: jpeg(x, rng)),
           ("glare", 0.2, lambda x: glare(x, rng, strength=rng.uniform(30, 90)))]
    for name, p, fn in ops:
        if name not in HELDOUT_FAMILIES and rng.random() < p:
            patch = fn(patch)
    return patch

class CellCrops(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def _view(self, it, rng, hf, vf):
        x0, y0, x1, y1 = it["box"]; bw, bh = x1 - x0, y1 - y0
        p = crop_lighting(it["patch"], rng, sigma=max(1.0, bw / 8))
        if it["code"] == REJECT:
            c = cut(p, it["box"], rng.uniform(-0.05, 0.25), rng.uniform(-0.1, 0.1) * bw, rng.uniform(-0.1, 0.1) * bh)
        else:
            c = cut(p, it["box"], rng.uniform(-0.05, 0.2), rng.uniform(-0.08, 0.08) * bw, rng.uniform(-0.08, 0.08) * bh)
        if hf: c = c[:, ::-1]
        if vf: c = c[::-1]
        return torch.from_numpy(normalize_crop(np.ascontiguousarray(c)))[None]
    def __getitem__(self, i):
        it = self.items[i]
        if not self.train:
            return torch.from_numpy(normalize_crop(cut(it["patch"], it["box"])))[None], it["code"]
        rng = np.random.default_rng()
        hf, vf = rng.random() < 0.5, rng.random() < 0.5
        y = it["code"]
        if hf: y = HMAP[y]
        if vf: y = VMAP[y]
        return self._view(it, rng, hf, vf), self._view(it, np.random.default_rng(), hf, vf), y

class SimpleBrailleCNN(nn.Module):  # identical to braille_cnn/cnn.py
    def __init__(self, num_classes=64, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(4), nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def load_old_cnn():
    sd = torch.load(CNN_BASE, map_location="cpu")
    sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
    m = SimpleBrailleCNN(64); m.load_state_dict(sd, strict=True)
    return m.eval()

def expand_to_reject(old):
    new = SimpleBrailleCNN(NC)
    sd = old.state_dict()
    nsd = new.state_dict()
    for k, v in sd.items():
        if k == "classifier.5.weight":
            nsd[k][:64] = v; nn.init.normal_(nsd[k][64:], std=0.01)
        elif k == "classifier.5.bias":
            nsd[k][:64] = v; nsd[k][64] = v.min() - 1.0
        else:
            nsd[k] = v
    new.load_state_dict(nsd)
    return new

counts = np.bincount([i["code"] for i in ITEMS["train"]], minlength=NC).astype(np.float64)
w_item = np.array([1.0 / np.sqrt(max(counts[i["code"]], 1)) for i in ITEMS["train"]])
sampler = WeightedRandomSampler(torch.tensor(w_item), num_samples=len(w_item), replacement=True)
train_dl = DataLoader(CellCrops(ITEMS["train"], True), batch_size=CNN_BATCH, sampler=sampler, num_workers=2, drop_last=True)

def eval_items(split, family=None, sev=None):
    """Crops cut exactly like the app (margin 0) from clean or corrupted pages."""
    xs, ys = [], []
    if family is None:
        for it in ITEMS[split]:
            xs.append(normalize_crop(cut(it["patch"], it["box"]))); ys.append(it["code"])
        return torch.tensor(np.stack(xs))[:, None], torch.tensor(ys)
    for rec in split_pages(split):
        img = corrupt(rec["img"], family, sev, 1000 * sev + rec["page"], PAGE_SIGMA)
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        for b, c in zip(rec["boxes"], rec["codes"]):
            x0, y0, x1, y1 = [int(round(v)) for v in b]
            xs.append(normalize_crop(cv2.resize(g[y0:y1, x0:x1], (64, 64), interpolation=cv2.INTER_CUBIC))); ys.append(int(c))
    return torch.tensor(np.stack(xs))[:, None], torch.tensor(ys)

VAL_CLEAN = eval_items("val")
VAL_CORR = [eval_items("val", f, 2) for f in FAMILIES if f not in HELDOUT_FAMILIES]
dev = torch.device("cuda")

@torch.no_grad()
def logits_of(model, x, tta=False):
    """With tta: average softmax over 4 flips, mapping each flipped prediction back through the dot remap."""
    model.eval(); out = []
    nc = model.classifier[-1].out_features
    idx = lambda mp: torch.tensor(mp[:nc])
    for i in range(0, len(x), 1024):
        xb = x[i:i + 1024].to(dev)
        if not tta:
            out.append(model(xb).float().cpu()); continue
        p = F.softmax(model(xb), 1)
        p = p + F.softmax(model(xb.flip(3)), 1)[:, idx(HMAP)]
        p = p + F.softmax(model(xb.flip(2)), 1)[:, idx(VMAP)]
        p = p + F.softmax(model(xb.flip(2).flip(3)), 1)[:, idx([VMAP[h] for h in HMAP])]
        out.append(torch.log(p / 4).cpu())
    return torch.cat(out)

def cell_metrics(model, x, y, tta=False):
    pred = logits_of(model, x, tta).argmax(1)
    is_cell = y != REJECT
    rejected = (pred == REJECT) | (pred == 0) | torch.tensor([int(p) not in MAPPED for p in pred])
    return dict(cell_acc=float((pred[is_cell] == y[is_cell]).float().mean()) if is_cell.any() else float("nan"),
                cells_dropped=float(rejected[is_cell].float().mean()) if is_cell.any() else float("nan"),
                not_cell_rejected=float(rejected[~is_cell].float().mean()) if (~is_cell).any() else float("nan"))

model = expand_to_reject(load_old_cnn()).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=CNN_EPOCHS * len(train_dl), pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")
LAMBDA_JSD = 2.0
best_score, CNN_NEW = -1, OUT / "braille_cnn_lighting.pt"

for ep in range(1, CNN_EPOCHS + 1):
    model.train(); t0 = time.time(); tot = 0
    for v1, v2, y in train_dl:
        v1, v2, y = v1.to(dev), v2.to(dev), y.to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            l1, l2 = model(v1), model(v2)
            ce = 0.5 * (F.cross_entropy(l1, y, label_smoothing=0.05) + F.cross_entropy(l2, y, label_smoothing=0.05))
            # log space: the old CNN's logits reach ~90, so softmax underflows to 0 and log(0) gives NaN gradients
            lp1, lp2 = F.log_softmax(l1.float(), 1), F.log_softmax(l2.float(), 1)
            lm = torch.logsumexp(torch.stack([lp1, lp2]), 0) - math.log(2)
            jsd = 0.5 * (F.kl_div(lm, lp1, reduction="batchmean", log_target=True)
                         + F.kl_div(lm, lp2, reduction="batchmean", log_target=True))
            loss = ce + LAMBDA_JSD * jsd
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += float(loss)
    clean = cell_metrics(model, *VAL_CLEAN)
    corr = np.mean([cell_metrics(model, *vc)["cell_acc"] for vc in VAL_CORR])
    score = 0.4 * clean["cell_acc"] + 0.4 * corr + 0.2 * clean["not_cell_rejected"]
    print(f"ep {ep:02d} loss {tot / len(train_dl):.3f} | val cell acc {clean['cell_acc']:.3f} "
          f"corrupted {corr:.3f} | not-a-cell rejected {clean['not_cell_rejected']:.3f} | {time.time() - t0:.0f}s")
    if score > best_score:
        best_score = score; torch.save(model.state_dict(), CNN_NEW)
print("saved", CNN_NEW)

## 10) Evaluate classifiers: old vs new
`cell_acc` = correct dot code on real cells. `cells_dropped` = real cells the app would discard (predicted blank,
not-a-cell, or a code without a Sinhala label). `not_cell_rejected` = non-cells correctly discarded (higher is better).
TTA = average over the 4 flips with dot-remapping (can be added to the app later; ~4× classifier time).

In [ ]:
old_cnn = load_old_cnn().to(dev)
new_cnn = SimpleBrailleCNN(NC); new_cnn.load_state_dict(torch.load(CNN_NEW)); new_cnn = new_cnn.to(dev).eval()

TEST_SETS = {"clean (high+low-light photos)": eval_items("test")}
for f in FAMILIES:
    for s in (1, 3):
        TEST_SETS[f"{f} s{s}"] = eval_items("test", f, s)
cls_rows = []
for name, (x, y) in TEST_SETS.items():
    r = {"set": name}
    for tag, mdl, tta in (("old", old_cnn, False), ("new", new_cnn, False), ("new+TTA", new_cnn, True)):
        mt = cell_metrics(mdl, x, y, tta)
        r[f"{tag} cell_acc"] = round(mt["cell_acc"], 3); r[f"{tag} dropped"] = round(mt["cells_dropped"], 3)
        if name.startswith("clean"):
            r[f"{tag} not_cell_rejected"] = round(mt["not_cell_rejected"], 3)
    cls_rows.append(r)
cls_table = pd.DataFrame(cls_rows).set_index("set")
display(cls_table)
cls_table.to_csv(OUT / "classifier_comparison.csv")

# Real low-light photos only (the most honest number: real, not synthetic)
real_low = [r for r in split_pages("test") if r["variant"] == "low"]
if real_low:
    xs, ys = [], []
    for rec in real_low:
        g = cv2.cvtColor(rec["img"], cv2.COLOR_BGR2GRAY)
        for b, c in zip(rec["boxes"], rec["codes"]):
            x0, y0, x1, y1 = [int(round(v)) for v in b]
            xs.append(normalize_crop(cv2.resize(g[y0:y1, x0:x1], (64, 64), interpolation=cv2.INTER_CUBIC))); ys.append(int(c))
    xl, yl = torch.tensor(np.stack(xs))[:, None], torch.tensor(ys)
    print("REAL low-light test photos -> old:", cell_metrics(old_cnn, xl, yl), "\n                              new:", cell_metrics(new_cnn, xl, yl))

# Confidence threshold for an "uncertain - rescan" message: lowest max-probability where accepted val cells are >= 97% right
with torch.no_grad():
    pv = F.softmax(logits_of(new_cnn, VAL_CLEAN[0]), 1)
conf, pred = pv.max(1); yv = VAL_CLEAN[1]; cell = yv != REJECT
UNCERTAIN_THR = 0.0
for t in np.arange(0.3, 0.96, 0.05):
    acc = (conf >= t) & cell
    if acc.sum() and float((pred[acc] == yv[acc]).float().mean()) >= 0.97:
        UNCERTAIN_THR = float(round(t, 2)); break
print(f"suggested 'uncertain' threshold on softmax max-prob: {UNCERTAIN_THR} "
      f"(keeps {float(((conf >= UNCERTAIN_THR) & cell).sum() / cell.sum()):.1%} of val cells)")

## 11) End-to-end: how many cells appear on a page vs on a Braille-free image?
Detector (own validation threshold) → crop → classifier → the app's drop rule. On Braille-free images the ideal is **0**.

In [ ]:
def end_to_end(det_name, cnn, images):
    kept_counts = []
    for (b, s), im in zip(predict(DET_MODELS[det_name], images), images):
        b = b[s >= DET_THR[det_name]]
        if len(b) == 0:
            kept_counts.append(0); continue
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        xs = []
        for x0, y0, x1, y1 in b.round().astype(int):
            x0, y0 = max(x0, 0), max(y0, 0); x1, y1 = max(x1, x0 + 1), max(y1, y0 + 1)
            xs.append(normalize_crop(cv2.resize(g[y0:y1, x0:x1], (64, 64), interpolation=cv2.INTER_CUBIC)))
        pred = logits_of(cnn, torch.tensor(np.stack(xs))[:, None]).argmax(1)
        kept_counts.append(int(sum(int(p) in MAPPED for p in pred)))
    return np.array(kept_counts)

neg_imgs = [im for _, im, _ in EVAL["neg"]]
page_imgs = [im for _, im, _ in EVAL["clean"]]
gt_cells = [len(g) for _, _, g in EVAL["clean"]]
e2e = {}
for tag, dn, cnn in (("old det + old cnn", "old", old_cnn), ("new det + new cnn", "new", new_cnn)):
    neg_k, page_k = end_to_end(dn, cnn, neg_imgs), end_to_end(dn, cnn, page_imgs)
    e2e[tag] = {"Braille-free imgs with >=1 cell": f"{int((neg_k > 0).sum())}/{len(neg_k)}",
                "mean cells on Braille-free img": round(float(neg_k.mean()), 2),
                "cells kept on test pages": f"{int(page_k.sum())} (labelled: {sum(gt_cells)})"}
display(pd.DataFrame(e2e).T)

## 12) Visual check on raw glasses-camera frames (unlabelled)

In [ ]:
real = sorted((WORK / "real_unlabeled").glob("*.jpg"))[::5][:6]
if real:
    ims = [cv2.imread(str(p)) for p in real]
    panels = []
    for name in ("old", "new"):
        for (b, s), im in zip(predict(DET_MODELS[name], ims), ims):
            vis = im.copy()
            for x0, y0, x1, y1 in b[s >= DET_THR[name]].astype(int):
                cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 255, 255) if name == "new" else (0, 0, 255), 2)
            cv2.putText(vis, f"{name}: {int((s >= DET_THR[name]).sum())}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
            panels.append(cv2.resize(vis, (640, int(640 * vis.shape[0] / vis.shape[1]))))
    half = len(panels) // 2
    sheet = np.vstack([np.hstack(panels[:half]), np.hstack(panels[half:])])
    cv2.imwrite(str(OUT / "glasses_frames_old_vs_new.jpg"), sheet)
    display(_Img(str(OUT / "glasses_frames_old_vs_new.jpg"), width=1400))
    print("top row: old detector (red) | bottom row: new detector (yellow)")

## 13) Export ONNX files for the app (not switched — copy manually only if the tables above say "better")

In [ ]:
import onnx
from onnxruntime.quantization import QuantType, quantize_dynamic
import onnxruntime as ort

EXP = OUT / "export"; EXP.mkdir(exist_ok=True)

# Detector — same flags as cell_detect/export_to_onnx.py
onnx_path = Path(YOLO(str(DET_NEW)).export(format="onnx", imgsz=DET_IMGSZ, simplify=True, opset=12, dynamic=False,
                                            half=False, nms=False, batch=1, device="cpu", max_det=800))
det_fp32 = EXP / "braille_cell_yolo26n_lighting.onnx"; shutil.copy2(onnx_path, det_fp32)
m = onnx.load(str(det_fp32))
if m.ir_version > 9:
    m.ir_version = 9; onnx.save_model(m, str(det_fp32))
det_mobile = EXP / "braille_cell_yolo26n_lighting_mobile.onnx"
quantize_dynamic(str(det_fp32), str(det_mobile), weight_type=QuantType.QUInt8)
out_shape = [d.dim_value for d in onnx.load(str(det_fp32)).graph.output[0].type.tensor_type.shape.dim]
(EXP / "braille_cell_yolo26n_lighting_meta.json").write_text(json.dumps({
    "model": det_fp32.name, "task": "detect", "class_names": ["braille_cell"], "num_classes": 1, "imgsz": DET_IMGSZ,
    "input_layout": "NCHW", "color_format": "RGB", "normalize": {"scale": 1 / 255, "mean": [0, 0, 0], "std": [1, 1, 1]},
    "conf_threshold": DET_THR["new"], "iou_threshold": 0.40, "max_det": 800, "nms_in_app": True,
    "outputs": [{"name": "output0", "shape": out_shape}], "source_weights": "braille_cell_lighting.pt",
    "provenance_notes": "Fine-tuned from braille_cell_gold_degraded.pt with lighting augmentation (relief relighting, "
                        "directional light, shadows, low light, blur, glasses softening, JPEG) and Braille-free negatives. "
                        "conf_threshold chosen on validation pages + negatives.",
}, indent=2))

# Classifier — same contract as braille_cnn/export_onnx.py (input 'input' [N,1,64,64], output 'logits'), now 65 classes
cnn_cpu = SimpleBrailleCNN(NC); cnn_cpu.load_state_dict(torch.load(CNN_NEW, map_location="cpu")); cnn_cpu.eval()
cnn_onnx = EXP / "braille_cnn_lighting.onnx"
torch.onnx.export(cnn_cpu, torch.zeros(1, 1, 64, 64), str(cnn_onnx), input_names=["input"], output_names=["logits"],
                  opset_version=17, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, dynamo=False)
m = onnx.load(str(cnn_onnx))
if m.ir_version > 9:
    m.ir_version = 9
onnx.save_model(m, str(cnn_onnx), save_as_external_data=False)
sess = ort.InferenceSession(str(cnn_onnx), providers=["CPUExecutionProvider"])
xt = VAL_CLEAN[0][:64].numpy()
agree = (sess.run(None, {"input": xt})[0].argmax(1) == cnn_cpu(torch.from_numpy(xt)).argmax(1).numpy()).mean()
print(f"CNN ONNX parity: argmax agree {agree:.0%}")

summary = {
    "detector_threshold_val": DET_THR, "uncertain_threshold": UNCERTAIN_THR,
    "detector": det_table.reset_index().to_dict(orient="records"),
    "classifier": cls_table.reset_index().to_dict(orient="records"),
    "end_to_end": e2e, "page_gate": gate_rows,
    "config": dict(TRAIN_PAGES=TRAIN_PAGES, VAL_PAGES=VAL_PAGES, TEST_PAGES=TEST_PAGES, TRAIN_LOW_PAGES=TRAIN_LOW_PAGES,
                   DET_AUG_VARIANTS=DET_AUG_VARIANTS, DET_NEG_FRACTION=DET_NEG_FRACTION, HELDOUT_FAMILIES=HELDOUT_FAMILIES),
}
(OUT / "lighting_robust_summary.json").write_text(json.dumps(summary, indent=2, default=str))
for p in sorted(EXP.iterdir()):
    print(f"{p.name:45s} {p.stat().st_size / 1e6:6.2f} MB")
print("\nAll results in", OUT)

## 14) Bringing results back (only if better)
Download `outputs/` from Drive. **Switch a model only if** the new one wins on the clean test pages *and* the real
low-light photos *and* the corrupted sets, without more cells appearing on Braille-free images.

| File | Goes to | App change |
|---|---|---|
| `braille_cell_lighting.pt` | `cell_detect/weights/` | — (source checkpoint) |
| `braille_cell_yolo26n_lighting_mobile.onnx` + `_lighting.onnx` + `_meta.json` | `braille_lens_flutter/assets/models/` | point `cellDetectorOnnxAsset` / fallback at them; set `cellDetectorConfThreshold` to `conf_threshold` from the meta file |
| `braille_cnn_lighting.pt` | `braille_cnn/checkpoints/` | — |
| `braille_cnn_lighting.onnx` | `braille_lens_flutter/assets/models/` | point `brailleCnnAsset` at it. Class 64 has no label, so the app already drops it (`'#64'`). `braille_labels.json` stays the same. |

Optional app follow-ups measured here: the page-level gate (section 7), flip-TTA (section 10), and an "uncertain — rescan"
message below the suggested confidence threshold.